# NB06: Environmental Stratification

Tests whether P×Metal and N×Metal co-occurrence varies by reported isolation environment.
Uses `kbase_ke_pangenome.ncbi_env` metadata.

Run equivalent script: `python src/07_environmental_stratification.py`

**Last executed:** 2026-05-13  
**Git commit:** 6612ae0 (v4 soil+plant subset)  
**Environment:** BERDL JupyterHub / Python 3.10

In [1]:
import pandas as pd
import os
DATA_DIR = os.path.join('..', 'data')

In [2]:
env_map = pd.read_csv(os.path.join(DATA_DIR, 'env_species_mapping.csv'))
print(f'{len(env_map)} species with environment assignments')
print()
print('Species per environment:')
print(env_map['primary_env'].value_counts().to_string())

27017 species with environment assignments

Species per environment:
primary_env
other                    12628
human-associated          3497
marine                    3449
soil/rhizosphere          3409
freshwater/engineered     2059
plant-associated          1135
animal-associated          840


## Co-occurrence by Environment

In [3]:
env_cooc = pd.read_csv(os.path.join(DATA_DIR, 'env_cooccurrence.csv'))
for pair in ['P x Metal', 'N x Metal', 'Phz x Metal']:
    print(f'\n{pair}:')
    sub = env_cooc[env_cooc['pair']==pair].sort_values('enrichment', ascending=False)
    for _, r in sub.iterrows():
        sig = '*' if r['fisher_p'] < 0.05 else ''
        print(f"  {r['environment']:>25s}: n={r['n_species']:>5d}  "
              f"both={r['n_both']:>4d}  enrich={r['enrichment']:.2f}x  "
              f"log-OR={r['log_OR']:+.3f}  p={r['fisher_p']:.2e} {sig}")


P x Metal:
           human-associated: n= 3497  both=2643  enrich=1.02x  log-OR=+1.374  p=3.43e-20 *
                      other: n=12624  both=9403  enrich=1.02x  log-OR=+1.000  p=5.37e-39 *
      freshwater/engineered: n= 2059  both=1622  enrich=1.01x  log-OR=+1.308  p=6.55e-09 *
           plant-associated: n= 1134  both= 960  enrich=1.01x  log-OR=+1.512  p=1.25e-06 *
                     marine: n= 3449  both=2095  enrich=1.01x  log-OR=+0.259  p=7.84e-03 *
           soil/rhizosphere: n= 3406  both=2855  enrich=1.01x  log-OR=+1.343  p=1.93e-09 *
          animal-associated: n=  840  both= 494  enrich=1.01x  log-OR=+0.231  p=3.88e-01 

N x Metal:
                     marine: n= 3449  both= 461  enrich=1.22x  log-OR=+1.367  p=4.23e-23 *
          animal-associated: n=  840  both= 210  enrich=1.09x  log-OR=+2.268  p=2.08e-07 *
                      other: n=12624  both=2739  enrich=1.05x  log-OR=+1.490  p=1.71e-37 *
           human-associated: n= 3497  both= 713  enrich=1.03x  log-

In [4]:
from scipy.stats import rankdata
import numpy as np

p_values = env_cooc['fisher_p'].values
n_tests = len(p_values)
ranks = rankdata(p_values)
q_raw = p_values * n_tests / ranks
q_sorted_idx = np.argsort(p_values)[::-1]
q_adj = q_raw.copy()
for i in range(1, len(q_sorted_idx)):
    idx = q_sorted_idx[i]
    prev_idx = q_sorted_idx[i-1]
    if q_adj[idx] > q_adj[prev_idx]:
        q_adj[idx] = q_adj[prev_idx]
env_cooc['q_value'] = np.clip(q_adj, 0, 1)

print("BH FDR-corrected q-values (21 tests across 7 environments x 3 pairs):\n")
for pair in ['P x Metal', 'N x Metal', 'Phz x Metal']:
    print(f'{pair}:')
    sub = env_cooc[env_cooc['pair']==pair].sort_values('log_OR', ascending=False)
    for _, r in sub.iterrows():
        sig = '*' if r['q_value'] < 0.05 else ''
        print(f"  {r['environment']:>25s}: log-OR={r['log_OR']:+.3f}  "
              f"p={r['fisher_p']:.2e}  q={r['q_value']:.3f} {sig}")
    print()

BH FDR-corrected q-values (21 tests across 7 environments x 3 pairs):

P x Metal:
           plant-associated: log-OR=+1.512  p=1.25e-06  q=0.000 *
           human-associated: log-OR=+1.374  p=3.43e-20  q=0.000 *
           soil/rhizosphere: log-OR=+1.343  p=1.93e-09  q=0.000 *
      freshwater/engineered: log-OR=+1.308  p=6.55e-09  q=0.000 *
                      other: log-OR=+1.000  p=5.37e-39  q=0.000 *
                     marine: log-OR=+0.259  p=7.84e-03  q=0.015 *
          animal-associated: log-OR=+0.231  p=3.88e-01  q=0.543 

N x Metal:
          animal-associated: log-OR=+2.268  p=2.08e-07  q=0.000 *
                      other: log-OR=+1.490  p=1.71e-37  q=0.000 *
      freshwater/engineered: log-OR=+1.461  p=1.45e-05  q=0.000 *
                     marine: log-OR=+1.367  p=4.23e-23  q=0.000 *
           human-associated: log-OR=+0.968  p=7.08e-06  q=0.000 *
           plant-associated: log-OR=+0.940  p=6.41e-02  q=0.112 
           soil/rhizosphere: log-OR=+0.364  p=1.69

## Key finding

P×Metal is strongest in plant-associated and soil/rhizosphere environments,
consistent with the Fe-oxyhydroxide mineral surface hypothesis.

## Environmental Stratification Figure (Figure 3)

In [5]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

FIG_DIR = os.path.join('..', 'figures')

pairs = ['P x Metal', 'N x Metal']
envs_order = ['soil/rhizosphere', 'plant-associated', 'marine', 'freshwater/engineered',
              'human-associated', 'animal-associated']

fig, ax = plt.subplots(figsize=(10, 6))
bar_width = 0.35
y_pos = np.arange(len(envs_order))

for i, pair in enumerate(pairs):
    sub = env_cooc[env_cooc['pair'] == pair].set_index('environment')
    log_ors = [sub.loc[e, 'log_OR'] if e in sub.index else 0 for e in envs_order]
    ses = [sub.loc[e, 'se'] if e in sub.index else 0 for e in envs_order]
    pvals = [sub.loc[e, 'fisher_p'] if e in sub.index else 1 for e in envs_order]
    colors = []
    for p in pvals:
        if pair == 'P x Metal':
            colors.append('#2166ac' if p < 0.05 else '#aec7e8')
        else:
            colors.append('#b2182b' if p < 0.05 else '#f4a582')
    offset = -bar_width/2 + i * bar_width
    ax.barh(y_pos + offset, log_ors, bar_width,
            xerr=[np.array(log_ors) - (np.array(log_ors) - 1.96*np.array(ses)),
                  1.96*np.array(ses)],
            color=colors, edgecolor='none', alpha=0.85,
            error_kw=dict(ecolor='#333333', capsize=2, linewidth=0.8),
            label=pair)

ax.axvline(x=0, color='black', linewidth=0.8, linestyle='--')
n_species = []
for e in envs_order:
    row = env_cooc[(env_cooc['environment'] == e) & (env_cooc['pair'] == 'P x Metal')]
    n_species.append(int(row['n_species'].iloc[0]) if len(row) > 0 else 0)
ax.set_yticks(y_pos)
ax.set_yticklabels([f"{e} (n={n})" for e, n in zip(envs_order, n_species)], fontsize=9)
ax.set_xlabel('log Odds Ratio', fontsize=11)
ax.set_title('Environmental Stratification of Nutrient x Metal Co-occurrence', fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'figure3_env_stratification.png'), dpi=200, bbox_inches='tight')
plt.show()
print("Saved figures/figure3_env_stratification.png")

Saved figures/figure3_env_stratification.png
